In [1]:
import polars as pl 
from pathlib import Path
from yaml import safe_load
import seaborn as sns 
import numpy as np

sns.set_style('whitegrid')

In [2]:
res_path = Path(
    "/projectnb/vkolagrp/bellitti/adrd-foundation-model/adrd_simplified_evaluation/results/training_curve"
)

cols_to_read = ["ID", "ground_truth", "prediction"]


def load_data(res_path):

    dfs = []

    for fpath in res_path.rglob("*.parquet"):

        config_path = fpath.parent / 'config.yml'
        with config_path.open() as config_file:
            config = safe_load(config_file)

        df = pl.read_parquet(fpath, columns=cols_to_read).with_columns(
                pl.lit(fpath.parent.parent.name).alias("benchmark"),
                pl.lit(fpath.parent.parent.parent.name).alias("cohort"),
                (pl.col("ground_truth") == pl.col("prediction"))
                .cast(pl.Int8)
                .alias("correct"),
                pl.lit(config['training_steps']).alias('training_steps'),
                pl.lit(config['run_readable_name']).alias('model')
            )

        dfs.append(df)


    return pl.concat(dfs)

In [35]:
df = load_data(res_path)

In [36]:
df = df.filter(pl.col('benchmark').is_in(['test_mci','test_np_mixed','test_np']).not_())

In [37]:
df.select(pl.col('benchmark').unique())

benchmark
str
"""test_etpr"""
"""test_ftld"""
"""test_dat"""
"""test_cog"""
"""test_csf"""
"""test_np_one"""
"""test_pet"""


In [38]:
df = df.with_columns_seq(
    pl.when(pl.col("model").str.contains("3B"))
    .then(3)
    .when(pl.col("model").str.contains("7B"))
    .then(7)
    .otherwise(None).alias('params'),
)

In [39]:
df

ID,ground_truth,prediction,benchmark,cohort,correct,training_steps,model,params
str,str,str,str,str,i8,i32,str,i32
"""sub-CLB00196""","""A""","""A""","""test_cog""","""brainlat_test""",1,0,"""Qwen2.5-7B-Instruct""",7
"""sub-CLB00196""","""A""","""A""","""test_cog""","""brainlat_test""",1,0,"""Qwen2.5-7B-Instruct""",7
"""sub-CLB00196""","""A""","""A""","""test_cog""","""brainlat_test""",1,0,"""Qwen2.5-7B-Instruct""",7
"""sub-CLB00196""","""A""","""A""","""test_cog""","""brainlat_test""",1,0,"""Qwen2.5-7B-Instruct""",7
"""sub-CLB00196""","""A""","""A""","""test_cog""","""brainlat_test""",1,0,"""Qwen2.5-7B-Instruct""",7
…,…,…,…,…,…,…,…,…
"""131120""","""B""","""B""","""test_dat""","""ppmi_test""",1,1000,"""NACC-3B-OS-SCE-1000""",3
"""131120""","""B""","""B""","""test_dat""","""ppmi_test""",1,1000,"""NACC-3B-OS-SCE-1000""",3
"""131120""","""B""","""A""","""test_dat""","""ppmi_test""",0,1000,"""NACC-3B-OS-SCE-1000""",3


In [42]:
acc = df.filter(pl.col('model').str.contains('NACC')).group_by('training_steps','cohort','benchmark','params').agg(pl.col('correct').mean().alias('accuracy')).sort('training_steps')

acc = acc.with_columns_seq(
    pl.when(pl.col('cohort').str.contains('nacc')).then(pl.lit('Internal validation\n(NACC)')).otherwise(pl.lit('External validation\n(All other cohorts)')).alias('in_distribution'),
)

acc

training_steps,cohort,benchmark,params,accuracy,in_distribution
i32,str,str,i32,f64,str
100,"""nacc_test_updated""","""test_np_one""",3,0.535638,"""Internal validation (NACC)"""
100,"""nacc_test_updated""","""test_csf""",3,0.599306,"""Internal validation (NACC)"""
100,"""adni_test""","""test_pet""",3,0.632,"""External validation (All other…"
100,"""ppmi_test""","""test_etpr""",3,0.541935,"""External validation (All other…"
100,"""brainlat_test""","""test_cog""",3,0.479091,"""External validation (All other…"
…,…,…,…,…,…
1491,"""ppmi_test""","""test_cog""",3,0.7682,"""External validation (All other…"
1491,"""nifd_test""","""test_cog""",3,0.639041,"""External validation (All other…"
1491,"""adni_test""","""test_etpr""",3,0.6476,"""External validation (All other…"


In [47]:
base3 = (
    df.filter(pl.col("model") == "Qwen2.5-3B-Instruct")
    .group_by("cohort", "benchmark")
    .agg(
        pl.col("correct").mean().alias("accuracy"),
        (pl.col("correct")/pl.col('params')).mean().alias("accuracy_per_b"),
    )
)

n_boot = 1000
q3b_bsamples = np.array([base3.sample(fraction=1,with_replacement=True).select(pl.col('accuracy').mean()).item() for _ in range(n_boot)])

q3b_mean = q3b_bsamples.mean()
q3b_low = np.quantile(q3b_bsamples,0.025)
q3b_high = np.quantile(q3b_bsamples,0.975)

In [48]:
base3

cohort,benchmark,accuracy,accuracy_per_b
str,str,f64,f64
"""nacc_test_updated""","""test_dat""",0.735211,0.24507
"""ppmi_test""","""test_dat""",0.3742,0.124733
"""nifd_test""","""test_ftld""",0.410959,0.136986
"""adni_test""","""test_etpr""",0.5004,0.1668
"""brainlat_test""","""test_cog""",0.450303,0.150101
…,…,…,…
"""nacc_test_updated""","""test_pet""",0.62896,0.209653
"""nacc_test_updated""","""test_etpr""",0.4726,0.157533
"""nacc_test_updated""","""test_cog""",0.6592,0.219733


In [49]:
base7 = (
    df.filter(pl.col("model") == "Qwen2.5-7B-Instruct")
    .group_by("cohort", "benchmark")
    .agg(
        pl.col("correct").mean().alias("accuracy"),
        (pl.col("correct")/pl.col('params')).mean().alias("accuracy_per_b"),
    )
)

n_boot = 1000
q7b_bsamples = np.array([base7.sample(fraction=1,with_replacement=True).select(pl.col('accuracy').mean()).item() for _ in range(n_boot)])

q7b_mean = q7b_bsamples.mean()
q7b_low = np.quantile(q7b_bsamples,0.025)
q7b_high = np.quantile(q7b_bsamples,0.975)

In [50]:
base7

cohort,benchmark,accuracy,accuracy_per_b
str,str,f64,f64
"""nacc_test_updated""","""test_dat""",0.774648,0.110664
"""nifd_test""","""test_cog""",0.618493,0.088356
"""ppmi_test""","""test_cog""",0.7642,0.109171
"""nacc_test_updated""","""test_etpr""",0.61,0.087143
"""nifd_test""","""test_ftld""",0.64726,0.092466
…,…,…,…
"""nifd_test""","""test_etpr""",0.391781,0.055969
"""adni_test""","""test_etpr""",0.6688,0.095543
"""ppmi_test""","""test_etpr""",0.539427,0.077061
